<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
MCP Local
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q fastmcp langchain-mcp-adapters

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M30-MCP-Integration"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---


<p><font color='black' size="5">
@tool vs. MCP — wann welches Muster?
</font></p>



Agenten können Tools auf zwei grundlegend verschiedene Weisen nutzen:

| Kriterium | `@tool` (lokal) | MCP (remote) |
|-----------|----------------|-------------|
| **Laufzeit** | Im Jupyter-Kernel | Separater Prozess / Server |
| **Wiederverwendung** | Nur im aktuellen Notebook | Jede App, jedes LLM |
| **Sprache** | Nur Python | Python, Node.js, beliebig |
| **Skalierung** | Einfache Prototypen | Production-Services |
| **Sicherheit** | Direkter Code-Zugriff | Isolierter Prozess |
| **Protokoll** | LangChain intern | JSON-RPC 2.0 über HTTP |

**Faustregel:**

`@tool` für schnelle Prototypen und Kurs-Demos — MCP wenn Tools wiederverwendbar, sprachunabhängig oder produktionsreif sein sollen.


# 2 | Model Context Protocol (MCP)
---

<p><font color='black' size="5">Was ist MCP?</font></p>

Ein Protokoll ist ein Regelwerk, das bestimmt, wie zwei Systeme miteinander kommunizieren. Protokolle regeln die Datenübertragung in Computernetzwerken, bei der Internetkommunikation und zwischen Softwaresystemen.

**Zum Beispiel:**

- **HTTP** (Hypertext Transfer Protocol): Ermöglicht Websites die Kommunikation mit Browsern.
- **TCP/IP**: Definiert, wie Datenpakete im Internet geroutet werden.
- **JSON-RPC**: Ein Protokoll, das den Datenaustausch im JSON-Format ermöglicht.

Das **Model Context Protocol (MCP)** ist ein offenes Protokoll, das es großen Sprachmodellen (LLMs) ermöglicht, sich auf standardisierte Weise in externe Datenquellen und Tools zu integrieren. Dieses von Anthropic entwickelte Protokoll macht es KI-Modellen leicht, nahtlos mit einer Vielzahl von Tools und Datenquellen zusammenzuarbeiten — **ohne dass für jede Quelle spezifische API-Integrationen erforderlich sind**.





<p><font color='black' size="5">Warum wurde MCP entwickelt?</font></p>

Die Notwendigkeit von MCP ergibt sich aus den Ineffizienzen aktueller KI-API-Interaktionen. Derzeit ist der Aufbau von KI-Agenten, die Daten aus verschiedenen Quellen abrufen, **fragmentiert, repetitiv und schwer zu skalieren**. Jedes Tool spricht seine eigene Sprache und erfordert individuelle Integrationen. MCP reduziert diese Komplexität und minimiert den Entwicklungsaufwand.

```
Agent  →  MCP Client  →  MCP Server  →  Externe Ressource (DB, API, Filesystem)
       ←              ←              ←  Ergebnis
```

> Einmal als MCP-Server implementiert, funktioniert ein Tool mit **jedem kompatiblen LLM-Client** — Claude, GPT, LangGraph, u.a.

**Wichtige Ressourcen:**
- [Anthropic MCP](https://www.anthropic.com/news/model-context-protocol) — Offizielle Ankündigung und Spezifikation
- [MCP Server Gallery](https://github.com/modelcontextprotocol/servers) — Verfügbare Open-Source-Server

MCP ist ein **standardisiertes Protokoll**, das LLMs mit externen Tools verbindet — wie ein **USB-Standard für AI**. Einmal implementiert, funktioniert ein MCP-Tool mit jedem kompatiblen LLM-Client.

**Vier Kernkomponenten:**

| Komponente | Aufgabe | Beispiele |
|------------|---------|-----------|
| **MCP Host** | Die Anwendung, die den Client enthält und den Agenten ausführt | Chat-App, IDE-Assistent, Jupyter Notebook |
| **MCP Client** | Protokoll-Komponente innerhalb des Hosts — verbindet sich zu Servern, lädt Tool-Definitionen | `MultiServerMCPClient` |
| **MCP Server** | Stellt Tools bereit, implementiert JSON-RPC 2.0 — sprachunabhängig | FastMCP (Python), Node.js, beliebig |
| **LLM / Agent** | Entscheidet welche Tools gebraucht werden, orchestriert die Aufrufe | LangGraph ReAct-Agent |

> Ein Host kann mehrere Clients enthalten, ein Client kann sich zu mehreren Servern verbinden.


**Transport-Optionen:**

| Transport | Beschreibung | Einsatz |
|-----------|-------------|---------|
| `streamable_http` | HTTP POST mit JSON-RPC 2.0 | **Web-Services, Notebooks** ✅ |
| `stdio` | Standard Input/Output | Lokale Prozesse (Claude Desktop) |
| `sse` | Server-Sent Events | Echtzeit-Streaming vom Server |
| `websocket` | Bidirektionale Verbindung | Hochperformante Anwendungen |

> Dieses Modul verwendet `streamable_http` — ideal für Jupyter und Web-Deployments.

**Ablauf aus Nutzerperspektive** (Ergänzung zum technischen Protokoll-Ablauf in Section 3):

```
User:   "Wie viele Kunden habe ich?"

  1  Host fragt MCP Server:     Welche Tools sind verfügbar?
  2  Server antwortet:          [sql_abfrage, tabellen_liste, ...]
  3  Host → LLM:                Frage + verfügbare Tools
  4  LLM → Host:                "Nutze sql_abfrage('SELECT COUNT(*) FROM kunden')"
  5  Host → MCP Server:         Tool-Aufruf mit Parametern
  6  Server → Datenbank:        SQL-Abfrage ausführen → Ergebnis zurück
  7  Server → Host → LLM:       Ergebnis ("42")
  8  LLM → User:                "Sie haben aktuell 42 Kunden."
```

Der Host verwaltet den gesamten Ablauf — das LLM entscheidet nur, *welches Tool* gebraucht wird.
Der MCP Server entscheidet, *wie* das Tool ausgeführt wird (DB, API oder lokaler Code).


<p><font color='black' size="5">
Lokaler vs. öffentlicher MCP-Server
</font></p>



MCP-Server können auf zwei grundlegend verschiedene Arten betrieben werden:

| Merkmal | Lokaler Server (dieses Modul)| HF Space Server |
|---------|---------------|--------------------------------|
| **Adresse** | `127.0.0.1:8001/mcp` | `https://ralf42-simple-mcp.hf.space/mcp` |
| **Erreichbarkeit** | Nur lokal im Jupyter-Kernel | Öffentlich, weltweit |
| **Lebensdauer** | Nur während der Notebook-Session | Permanent (schläft nach Inaktivität ein) |
| **Deployment** | `subprocess.Popen(...)` im Notebook | Einmalig auf HF Space deployen |
| **MCP-Protokoll** | ✅ identisch | ✅ identisch |
| **Transport** | `streamable_http` | `streamable_http` |

> **Das MCP-Protokoll ist in beiden Fällen identisch** — nur die Adresse ändert sich.
> `MultiServerMCPClient` verbindet sich genauso — nur mit einer anderen URL.

**Faustregel:**      
Lokaler Server für schnelle Prototypen und Kurs-Demos — HF Space wenn der Server dauerhaft und öffentlich zugänglich sein soll.

# 3 | Lokalen Server erstellen
---

In [ ]:
%%writefile math_mcp_server.py
from fastmcp import FastMCP

mcp = FastMCP('MathTools')

@mcp.tool()
def add(a: float, b: float) -> float:
    '''Addiert zwei Zahlen.'''
    return a + b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    '''Multipliziert zwei Zahlen.'''
    return a * b

@mcp.tool()
def subtract(a: float, b: float) -> float:
    '''Subtrahiert b von a.'''
    return a - b

@mcp.tool()
def divide(a: float, b: float) -> float:
    '''Teilt a durch b.'''
    if b == 0:
        raise ValueError('Division durch 0 nicht erlaubt')
    return a / b

if __name__ == '__main__':
    mcp.run(transport='http', host='127.0.0.1', port=8001, path='/mcp')

In [ ]:
# Server als subprocess starten
import subprocess, time, sys

mprint("### 🚀 Math-MCP-Server (Port 8001)\n---")

math_server_process = subprocess.Popen(
    [sys.executable, "math_mcp_server.py"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print("⏳ Starte Server...")

import requests
for i in range(12):
    try:
        r = requests.post("http://127.0.0.1:8001/mcp",
                          json={"jsonrpc":"2.0","method":"tools/list","id":1}, timeout=2)
        if r.status_code in [200, 400, 406]:
            break
    except Exception:
        time.sleep(1)

mprint("\n✅ Math-MCP-Server läuft auf `http://127.0.0.1:8001/mcp`")
print(f"   PID: {math_server_process.pid} | Tools: add, multiply, subtract, divide")

In [ ]:
# Server Health-Check
import requests

try:
    r = requests.get("http://127.0.0.1:8001/mcp", timeout=3)
    # Status 400/405/406 mit JSON = Server läuft korrekt
    if r.status_code in [200, 400, 405, 406]:
        print(f"✅ Server ONLINE — HTTP {r.status_code} (erwartet)")
    else:
        print(f"⚠️ Unerwarteter Status: {r.status_code}")
except Exception as e:
    print(f"❌ Server nicht erreichbar: {e}")

# 4 | MCP-Server für LangGraph-Agenten
---

Der `MultiServerMCPClient` aus `langchain-mcp-adapters` lädt die Tool-Definitionen vom Server und macht sie für LangGraph-Agenten verfügbar.

**Ablauf in zwei Phasen:**

```
Phase 1 — Handshake & Tool-Discovery:
  Client → Server: initialize
  Server → Client: Capabilities
  Client → Server: tools/list
  Server → Client: [add, multiply, subtract, divide]

Phase 2 — Tool-Ausführung (pro Agent-Schritt):
  Agent → Client: Tool-Call "add(3, 5)"
  Client → Server: HTTP POST JSON-RPC
  Server → Client: 8
  Agent bekommt: ToolMessage(content="8")
```

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  sequenceDiagram: Handshake + Tool Discovery</font> </br></p>

diagram = '''
sequenceDiagram
    autonumber
    actor User
    participant Agent as LangGraph Agent
    participant Client as MCP Client
    participant Server as FastMCP Server

    Note over Agent,Server: Phase 1 — Handshake & Tool Discovery
    Client->>Server: HTTP POST (initialize)
    Server-->>Client: Capabilities & Version
    Client->>Server: HTTP POST (tools/list)
    Server-->>Client: [add, multiply, subtract, divide]

    Note over Agent,Server: Phase 2 — Agent-Query
    User->>Agent: "Berechne (3+5) × 12"
    Agent->>Client: Tool: add(3, 5)
    Client->>Server: HTTP POST JSON-RPC
    Server-->>Client: 8
    Client-->>Agent: ToolMessage("8")
    Agent->>Client: Tool: multiply(8, 12)
    Client->>Server: HTTP POST JSON-RPC
    Server-->>Client: 96
    Client-->>Agent: ToolMessage("96")
    Agent->>User: "Das Ergebnis ist 96"
'''
mermaid(diagram, width=750)

<p><font color='black' size="5">
MCP Client + Tools laden
</font></p>

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client_math = MultiServerMCPClient({
    "math": {
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8001/mcp"
    }
})

math_tools = await client_math.get_tools()

print(f"✅ {len(math_tools)} Tools geladen:")
for t in math_tools:
    print(f"   • {t.name}: {t.description}")

<p><font color='black' size="5">
LangGraph ReAct-Agent mit MCP-Tools
</font></p>

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from IPython.display import Image as IPImage, display

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

math_agent = create_react_agent(
    llm,
    tools=math_tools,
    prompt=(
        "Du bist ein Mathe-Assistent mit Zugriff auf Rechen-Tools über MCP.\n"
        "Nutze die Tools für alle Berechnungen. Antworte auf Deutsch."
    ),
)

In [ ]:
display(IPImage(math_agent.get_graph(xray=True).draw_mermaid_png()))


<p><font color='black' size="5">
Demo: Einfache und komplexe Berechnungen
</font></p>

In [ ]:
queries = [
    "Berechne 127 × 8.",
    "Was ist (3 + 5) × 12?",
    "Berechne (100 - 20) / 4.",
]

for query in queries:
    result = await math_agent.ainvoke({"messages": [HumanMessage(query)]})
    mprint(f"**Query:** {query}\n\n**Antwort:** {result['messages'][-1].content}\n\n---")

# 5 | Custom MCP-Tools entwickeln
---

Jede Python-Funktion kann mit `@mcp.tool()` zu einem MCP-Tool werden. Wichtig: **Typ-Annotationen** und **Docstrings** sind für das LLM essentiell — sie beschreiben, wann und wie das Tool eingesetzt werden soll.

```python
@mcp.tool()
def erstelle_notiz(id: str, inhalt: str) -> str:
    """Erstellt eine neue Notiz mit der angegebenen ID."""
    notizen[id] = inhalt
    return f"Notiz {id!r} gespeichert."
```

**Design-Regeln für MCP-Tools:**

| Regel | Begründung |
|-------|----------|
| Eindeutige, beschreibende Namen | LLM wählt Tool anhand des Namens |
| Präzise Docstrings (1 Satz) | Wird direkt an das LLM übergeben |
| Typ-Annotationen für alle Parameter | Erzeugt automatisch JSON-Schema |
| Atomare Operationen | Ein Tool = eine Aufgabe |
| Fehlerbehandlung im Tool | LLM erhält lesbares Feedback |

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: Notiz-Server Architektur</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    A["LangGraph Agent"]
    C["MCP Client"]
    S["Notiz-Server\nPort 8002"]
    DB[("In-Memory\nnotizen = {}")]

    A -->|Tool Call| C
    C -->|HTTP POST| S
    S -->|lesen/schreiben| DB
    S -->|Ergebnis| C
    C -->|ToolMessage| A

    subgraph tools ["Tools: NotizTools"]
        T1["erstelle_notiz(id, inhalt)"]
        T2["lese_notiz(id)"]
        T3["liste_notizen()"]
        T4["loesche_notiz(id)"]
    end
    S --- tools

    style A  fill:#2E7D32,color:#fff
    style C  fill:#1565C0,color:#fff
    style S  fill:#37474F,color:#fff
    style DB fill:#4A148C,color:#fff
'''
mermaid(diagram, width=1250)


<p><font color='black' size="5">
Notiz-Server erstellen und starten
</font></p>


In [ ]:
%%writefile notiz_mcp_server.py
from fastmcp import FastMCP

mcp = FastMCP('NotizTools')
notizen = {}  # In-Memory Speicher

@mcp.tool()
def erstelle_notiz(id: str, inhalt: str) -> str:
    '''Erstellt oder überschreibt eine Notiz mit der angegebenen ID.'''
    notizen[id] = inhalt
    return f'Notiz {id!r} gespeichert ({len(inhalt)} Zeichen).'

@mcp.tool()
def lese_notiz(id: str) -> str:
    '''Liest den Inhalt einer Notiz anhand ihrer ID.'''
    return notizen.get(id, f'Notiz {id!r} nicht gefunden.')

@mcp.tool()
def liste_notizen() -> str:
    '''Listet alle gespeicherten Notiz-IDs und deren Anfang auf.'''
    if not notizen:
        return 'Keine Notizen vorhanden.'
    return '\n'.join(f'• {k}: {v[:40]}...' if len(v)>40 else f'• {k}: {v}' for k,v in notizen.items())

@mcp.tool()
def loesche_notiz(id: str) -> str:
    '''Löscht eine Notiz mit der angegebenen ID.'''
    if id in notizen:
        del notizen[id]
        return f'Notiz {id!r} gelöscht.'
    return f'Notiz {id!r} nicht gefunden.'

if __name__ == '__main__':
    mcp.run(transport='http', host='127.0.0.1', port=8002, path='/mcp')

In [ ]:
# Notiz-Server als subprocess starten
import subprocess, time, sys, requests

print("✅ notiz_mcp_server.py erstellt")

notiz_server_process = subprocess.Popen(
    [sys.executable, "notiz_mcp_server.py"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print("⏳ Starte Notiz-Server auf Port 8002...")

for i in range(12):
    try:
        r = requests.post("http://127.0.0.1:8002/mcp",
                          json={"jsonrpc":"2.0","method":"tools/list","id":1}, timeout=2)
        if r.status_code in [200, 400, 406]: break
    except Exception:
        time.sleep(1)

mprint("\n✅ Notiz-Server läuft auf http://127.0.0.1:8002/mcp")
print(f"   PID: {notiz_server_process.pid} | Tools: erstelle, lese, liste, loesche")


<p><font color='black' size="5">
Agent mit Notiz-Tools
</font></p>

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage

client_notiz = MultiServerMCPClient({
    "notiz": {
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8002/mcp"
    }
})
notiz_tools = await client_notiz.get_tools()
print(f"✅ {len(notiz_tools)} Notiz-Tools geladen: {[t.name for t in notiz_tools]}")

notiz_agent = create_react_agent(
    llm,
    tools=notiz_tools,
    prompt=(
        "Du bist ein Notiz-Assistent. Verwalte Notizen über MCP-Tools.\n"
        "Antworte kurz und auf Deutsch."
    ),
)

aufgaben = [
    "Erstelle eine Notiz 'einkauf' mit dem Inhalt: Milch, Brot, Eier, Käse.",
    "Erstelle eine Notiz 'meeting' mit dem Inhalt: Montag 10:00 Uhr, Thema MCP-Integration.",
    "Liste alle vorhandenen Notizen auf.",
    "Lese den Inhalt der Notiz 'einkauf'.",
    "Lösche die Notiz 'meeting' und bestätige die verbleibenden Notizen.",
]

for aufgabe in aufgaben:
    result = await notiz_agent.ainvoke({"messages": [HumanMessage(aufgabe)]})
    mprint(f"**Aufgabe:** {aufgabe}\n\n**Antwort:** {result['messages'][-1].content}\n\n---")

# 6 | Beispiele: Multi-Server, Filesystem, Datenbank
---


<p><font color='black' size="5">
Multi-Server Pattern
</font></p>

`MultiServerMCPClient` kann mehrere Server gleichzeitig verwalten. Der Agent sieht alle Tools als einheitliche Liste — er muss nicht wissen, von welchem Server ein Tool stammt.

```python
client = MultiServerMCPClient({
    "math":  {"transport": "streamable_http", "url": "http://127.0.0.1:8001/mcp"},
    "notiz": {"transport": "streamable_http", "url": "http://127.0.0.1:8002/mcp"},
})
alle_tools = await client.get_tools()  # add, multiply, ..., erstelle_notiz, ...
```


<p><font color='black' size="5">
Real-World Patterns
</font></p>



| Kategorie | Tools | Typische Bibliotheken |
|-----------|-------|-----------------------|
| **Filesystem** | `lese_datei`, `schreibe_datei`, `liste_verzeichnis` | `pathlib` |
| **SQLite** | `sql_abfrage`, `tabellen_liste`, `zeilen_zaehlen` | `sqlite3` |
| **Externe API** | `wetter_abfrage`, `waehrung_umrechnen` | `requests` |
| **Git** | `commit_log`, `diff_anzeigen` | `subprocess` + `git` |

> Jede Kategorie ist ein eigener FastMCP-Server — der Agent wählt automatisch das passende Tool.

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: Multi-Server Architektur</font> </br></p>

diagram = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    U(["👤 User"])
    A["LangGraph Agent\ngpt-4o-mini"]
    C["MultiServerMCPClient"]

    S1["MathTools\n:8001/mcp\nadd · multiply\nsubtract · divide"]
    S2["NotizTools\n:8002/mcp\nerstelle · lese\nliste · loesche"]

    U -->|komplexe Anfrage| A
    A -->|Tool-Auswahl| C
    C -->|HTTP POST| S1
    C -->|HTTP POST| S2
    S1 & S2 -->|Ergebnis| C
    C -->|ToolMessage| A
    A -->|kombinierte Antwort| U

    style A  fill:#2E7D32,color:#fff
    style C  fill:#1565C0,color:#fff
    style S1 fill:#37474F,color:#fff
    style S2 fill:#4A148C,color:#fff
    style U  fill:#E65100,color:#fff
'''
mermaid(diagram, width=900)

<p><font color='black' size="5">
Multi-Server Agent: Math + Notiz kombiniert
</font></p>


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage

# Beide Server in einem Client
client_multi = MultiServerMCPClient({
    "math":  {"transport": "streamable_http", "url": "http://127.0.0.1:8001/mcp"},
    "notiz": {"transport": "streamable_http", "url": "http://127.0.0.1:8002/mcp"},
})
alle_tools = await client_multi.get_tools()

zeilen = ["**Geladene Tools (beide Server):**", ""]
for t in alle_tools:
    zeilen.append(f"- `{t.name}`: {t.description}")
mprint("\n".join(zeilen))

multi_agent = create_react_agent(
    llm,
    tools=alle_tools,
    prompt=(
        "Du bist ein Assistent mit Zugriff auf Rechen-Tools und Notiz-Tools.\n"
        "Nutze beide Tool-Typen wo sinnvoll. Antworte auf Deutsch."
    ),
)

# Demo: Agent kombiniert beide Server in einer Anfrage
anfrage = (
    "Berechne 15% von 240 Euro und speichere das Ergebnis "
    "als Notiz 'rabatt'. Bestätige danach den Notizinhalt."
)
result = await multi_agent.ainvoke({"messages": [HumanMessage(anfrage)]})
mprint(f"**Anfrage:** {anfrage}\n\n**Antwort:** {result['messages'][-1].content}")


<p><font color='black' size="5">
Server-Cleanup
</font></p>

In [ ]:
mprint("### 🛑 MCP-Server beenden\n---")

for name, var in [("Math-Server", "math_server_process"),
                   ("Notiz-Server", "notiz_server_process")]:
    proc = globals().get(var)
    if proc:
        try:
            proc.terminate()
            proc.wait(timeout=5)
            print(f"✅ {name} (PID {proc.pid}) beendet")
        except Exception as e:
            print(f"⚠️ {name}: {e}")
    else:
        print(f"⚠️ {name}: Variable nicht gefunden")

import os
for f in ["math_mcp_server.py", "notiz_mcp_server.py"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑️ {f} gelöscht")

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M30-MCP-Integration", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.



<p><font color='black' size="5">
Eigene MCP-Tools entwickeln
</font></p>

**Aufgabe 1 — Erweiterter Math-Server:**
Füge dem Math-Server zwei neue Tools hinzu: `potenz(basis, exponent)` und `modulo(a, b)`. Teste mit: `"Berechne (5 hoch 3) modulo 7"` — der Agent soll beide Tools kombinieren.

**Aufgabe 2 — SQLite als MCP-Server:**
Erstelle einen `db_mcp_server.py` mit `sqlite3`, der diese Tools anbietet: `tabelle_erstellen(name, spalten)`, `zeile_einfuegen(tabelle, daten)`, `abfragen(tabelle)`. Verbinde einen Agenten damit.

**Aufgabe 3 — Drei-Server-Setup:**
Kombiniere Math-Server, Notiz-Server und deinen DB-Server in einem `MultiServerMCPClient`. Stelle eine Anfrage, die alle drei Server benötigt, z.B.: `"Berechne 12% von 500, speichere als Notiz und schreibe in die Datenbank."`

> 💡 **Tipp:** Mit `@mcp.tool(name="...")` kann der Tool-Name vom Funktionsnamen abweichen — nützlich wenn Funktionsnamen mit Python-Builtins kollidieren.